# How to Create a Note Alignment

This notebook demonstrates the essential pattern for aligning a performance
with a score using {{< glossary MatchClaim >}} objects in an
{{< glossary AlignmentBundle >}}.

**What you will learn:**

1. Match performance notes to score notes by shared attributes (pitch, staff)
2. Create an `AlignmentBundle` with performance and score groups
3. Query coordinates across both using `get_matchstamp_at()`
4. Create `MatchLine` objects from both directions
5. Export a `MatchLine` to the Vienna `.match` format
6. Opt into converted coordinates in a MatchStamp with `conversion_maps=True`

## TL;DR

```python
result = match_notes_by_attributes(perf_df, score_df, ["pitch", "staff"], ...)
bundle.add_match_claims(result.match_claims)
stamp = bundle.get_matchstamp_at(78.0, "clt1")  # quarterbeat 78 -> seconds
```

## 1. Setup

In [1]:
import tempfile

import pandas as pd

from timetoalign import Ms3Loader, TimelineGroup, TimeUnit
from timetoalign.alignment import AlignmentBundle, MatchLine
from timetoalign.alignment.match_format import MatchFileContext
from timetoalign.alignment.matching import (
    match_notes_by_attributes,
    prepare_abc_notes_for_matching,
    prepare_eep_notes_for_matching,
)
from timetoalign.loader.physical.eep_notes import EepNotesLoader
from timetoalign.testdata import ensure_data

DATA_DIR = ensure_data("score") / "beethoven_op18-4iv_multimodal"
NORMAL_DIR = DATA_DIR / "StringQuartetEEP_I_Normal"
ABC_DIR = DATA_DIR / "ABC"

/home/laser/miniconda3/envs/timetoalign/lib/python3.11/site-packages/partitura/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Load Performance & Score Notes

The performance notes come from `.notes` files (EEP format with timestamps
in seconds). The score notes come from a pre-unfolded TSV (with coordinates
in quarterbeats).

In [2]:
# Performance notes from the Normal recording
eep_loader = EepNotesLoader()
eep_loader.load(*sorted(NORMAL_DIR.glob("*_align_*.notes")))
eep_df = eep_loader.events.to_dataframe()

# Score notes from the unfolded ABC edition
abc_df = pd.read_csv(ABC_DIR / "n04op18-4_04_unfolded.notes.tsv", sep="\t")

{"EEP notes": len(eep_df), "ABC notes": len(abc_df)}

{'EEP notes': 4026, 'ABC notes': 3869}

## 3. Prepare & Match Notes

Before matching, we filter out rests and tied notes, and explode chords
into individual pitches.

In [3]:
eep_prepared = prepare_eep_notes_for_matching(eep_df)
abc_prepared = prepare_abc_notes_for_matching(abc_df)

{"EEP prepared": len(eep_prepared), "ABC prepared": len(abc_prepared)}

{'EEP prepared': 3756, 'ABC prepared': 3750}

Now match by pitch name and staff number. The matcher returns a `MatchResult`
containing the matched pairs and the generated `MatchClaim` objects.

In [4]:
match_result = match_notes_by_attributes(
    eep_prepared,
    abc_prepared,
    match_columns=["pitch", "staff"],
    source_coord_column="start",
    target_coord_column="quarterbeats_playthrough",
    source_timeline_id="cpt1",  # performance timeline (seconds)
    target_timeline_id="clt1",  # score timeline (quarterbeats)
    source_unit=TimeUnit.seconds,
    target_unit=TimeUnit.quarters,
)

match_result.summary()

{'matched': 3740,
 'unmatched_source': 16,
 'unmatched_target': 10,
 'match_claims': 3740}

## 4. Create AlignmentBundle

We need two {{< glossary TimelineGroup >}} objects: one for the performance,
one for the score. The `AlignmentBundle` holds both and manages cross-group
connections via {{< glossary MatchClaim >}} objects.

In [5]:
# Create the performance timeline (seconds)
perf_tl = eep_loader.create_timeline(uid="cpt1")

perf_group = TimelineGroup(
    id="performance",
    name="Normal Recording",
    timelines=[perf_tl],
)
perf_group

TimelineGroup(id='performance', n_timelines=1, n_timestamps=2, locked=False)

In [6]:
# Create the score timeline (quarterbeats)
score_loader = Ms3Loader.from_file(
    ABC_DIR / "n04op18-4_04.notes.tsv",
    ABC_DIR / "n04op18-4_04.measures.tsv",
)
clt1 = score_loader.create_timeline(uid="clt1")

score_group = TimelineGroup(
    id="score",
    name="ABC Score",
    timelines=[clt1],
)
score_group

TimelineGroup(id='score', n_timelines=1, n_timestamps=2, locked=False)

In [7]:
# Create the bundle and add the match claims
bundle = AlignmentBundle(name="Beethoven Op.18/4 — Simple Alignment")
bundle.add_group(perf_group)
bundle.add_group(score_group)
bundle.add_match_claims(match_result.match_claims)

bundle

AlignmentBundle(id='bundle:AlignmentBundle_1', name='Beethoven Op.18/4 — Simple Alignment', timelines=2, groups=2)

In [8]:
# Display an example MatchClaim (shows event IDs, timelines, coordinates)
match_result.match_claims[0]

MatchClaim(instant: cpt1@1.2 <-> clt1@1.0 [ANCHOR])

## 5. Query Coordinates via MatchStamp

The `get_matchstamp_at()` method is the primary interface for cross-group
coordinate transfer. Given a coordinate on one timeline, it returns
the corresponding coordinates on all connected timelines.

In [9]:
# Query from the score side: quarterbeat 78 (a matched note onset)
stamp = bundle.get_matchstamp_at(78.0, "clt1")
stamp

ID,Coordinate,Type
cpt1,18.218322,anchor
clt1,78,anchor


In [10]:
# The stamp shows coordinates on both timelines
{"score_qb": stamp.get_coordinate("clt1"), "perf_seconds": stamp.get_coordinate("cpt1")}

{'score_qb': Coordinate(78.0, quarters),
 'perf_seconds': Coordinate(18.218322, seconds)}

### Reverse lookup: performance to score

We can also query from the performance side. The claims store performance
coordinates in seconds (native EEP format).

In [11]:
# Find the score position for a performance coordinate (~100 seconds)
# Using 100.3583 which is an exact matched coordinate
stamp_rev = bundle.get_matchstamp_at(100.3583, "cpt1")

{
    "perf_seconds": stamp_rev.get_coordinate("cpt1"),
    "score_qb": stamp_rev.get_coordinate("clt1"),
}

{'perf_seconds': Coordinate(100.3583, seconds),
 'score_qb': Coordinate(417.0, quarters)}

## 6. Create MatchLines

A {{< glossary MatchLine >}} is an ordered sequence of coordinate pairs
for a given source timeline. It is the input for WarpMap generation.

The **direction matters**: the source timeline determines the ordering.

In [12]:
# Performance-to-score: source is performance, sorted by performance time
perf_to_score = MatchLine.from_claims(
    match_result.match_claims,
    source_timeline_id="cpt1",
)
perf_to_score

MatchLine(source='cpt1', stamps=1452, targets=[clt1])

In [13]:
# Score-to-performance: source is score, sorted by score position
score_to_perf = MatchLine.from_claims(
    match_result.match_claims,
    source_timeline_id="clt1",
)
score_to_perf

MatchLine(source='clt1', stamps=1452, targets=[cpt1])

**When to use which direction:**

- `perf_to_score`: Use when you have a performance coordinate and want to
  find the corresponding score position. Sorted by performance time.
- `score_to_perf`: Use when you have a score coordinate and want to find
  the corresponding performance time. Sorted by score position.

Both contain the same number of stamps (one per matched note), but the
ordering and lookup direction differ.

In [14]:
# Extract coordinate pairs for WarpMap construction
pairs = score_to_perf.get_coordinate_pairs("cpt1")

{
    "n_pairs": len(pairs),
    "first_pair": pairs[0],
    "last_pair": pairs[-1],
}

{'n_pairs': 1452, 'first_pair': (0.0, 1.0), 'last_pair': (1109.0, 264.916644)}

## 7. Export to .match Format

A {{< glossary MatchLine >}} can be exported to the Vienna `.match` file
format using `save_as()`.  The `.match` format is the standard interchange
format for note-level alignments in MIR.

To produce a rich `.match` file (with real pitch, duration, and staff
data rather than placeholders), supply a `MatchFileContext` built from
the same DataFrames used for matching.

In [15]:
ctx = MatchFileContext.from_dataframes(
    score_df=abc_prepared,
    perf_df=eep_prepared,
    match_result=match_result,
    piece="Beethoven Op.18/4-iv",
    composer="Ludwig van Beethoven",
    performer="StringQuartetEEP Normal",
)

with tempfile.TemporaryDirectory() as tmp:
    out_path = score_to_perf.save_as(f"{tmp}/alignment.match", context=ctx)
    text = out_path.read_text()

# Show the first 15 lines
for line in text.splitlines()[:15]:
    print(line)

info(matchFileVersion,1.0.0).
info(piece,Beethoven Op.18/4-iv).
info(composer,Ludwig van Beethoven).
info(performer,StringQuartetEEP Normal).
info(midiClockUnits,480).
info(midiClockRate,500000).
scoreprop(timeSignature,4/4,1:1,0,0.0000).
snote(n1,[E,b],5,1:1,0,1/2,0.0000,0.5000,[staff1])-note(n1482,75,960,1056,64,0,0).
snote(n2,[F,n],5,1:1,1/8,1/2,0.5000,1.0000,[staff1])-note(n1483,77,1056,1167,64,0,0).
snote(n3,[C,n],3,2:1,0,1,1.0000,2.0000,[staff4])-note(n0,48,1199,1311,64,0,0).
snote(n8,[E,b],5,2:1,1/8,1/2,1.5000,2.0000,[staff1])-note(n1485,75,1296,1399,64,0,0).
snote(n9,[F,n],5,2:1,1/4,1/2,2.0000,2.5000,[staff1])-note(n1486,77,1399,1488,64,0,0).
snote(n10,[D,n],5,2:1,3/8,1/2,2.5000,3.0000,[staff1])-note(n1487,74,1488,1599,64,0,0).
snote(n11,[C,n],3,2:1,1/2,1,3.0000,4.0000,[staff4])-note(n1488,75,1599,1695,64,0,0).
snote(n16,[C,n],5,2:1,5/8,1/2,3.5000,4.0000,[staff1])-note(n1489,72,1695,1799,64,0,0).


The exported file is a valid `.match` file that can be loaded back with
`MatchfileLoader` or any tool that reads the Vienna format.

## 8. Conversion Maps in MatchStamp Display

The score timeline already carries conversion maps from `Ms3Loader`
(ticks, floating measures), and a conversion map can be added to any
timeline at any time. Either way, `get_matchstamp_at()` leaves them out of
the {{< glossary MatchStamp >}} by default — `conversion_maps` is opt-in.

In [16]:
# Add one more conversion map, this time on the performance timeline.
from timetoalign.maps import ScalarMap

perf_tl.add_conversion_map(
    ScalarMap(
        scalar=1000, source_unit=TimeUnit.seconds, target_unit=TimeUnit.milliseconds
    )
)

In [17]:
# Default: conversion_maps=False, so none of the enabled maps show up.
bundle.get_matchstamp_at(78.0, "clt1")

ID,Coordinate,Type
cpt1,18.218322,anchor
clt1,78,anchor


In [18]:
# conversion_maps=True adds one row per enabled map, each evaluated at its
# own timeline's coordinate — ticks and floating measures from the score's
# loader-provided maps, milliseconds from the one just added.
bundle.get_matchstamp_at(78.0, "clt1", conversion_maps=True)

ID,Coordinate,Type
cpt1,18.218322,anchor
clt1,78,anchor
milliseconds,18218.322 milliseconds,cmap
ticks,37440 ticks,cmap
floating_measures,20.25 floating_measures,cmap


The same flag adds a derived column to `get_matchstamp_table()` — one
column per (timeline, enabled numeric-unit map), after the timeline
columns. A label-valued map would still show in the stamp display above
but never as a table column.

In [19]:
bundle.get_matchstamp_table(
    coordinates=[78.0], timeline_id="clt1", conversion_maps=True
).column_names

['clt1', 'cpt1', 'ticks', 'floating_measures', 'milliseconds']

## Summary

> *"MatchClaims connect timelines across groups. The AlignmentBundle
> manages these connections and provides coordinate transfer via
> MatchStamps. MatchLines order these stamps for WarpMap generation."*

| Pattern | API |
|---------|-----|
| Match notes by attributes | `match_notes_by_attributes()` |
| Add claims to bundle | `bundle.add_match_claims(claims)` |
| Query by coordinate | `bundle.get_matchstamp_at(coord, tl_id)` |
| Create MatchLine | `MatchLine.from_claims(claims, source_timeline_id)` |
| Export to `.match` | `matchline.save_as("out.match", context=ctx)` |
| Reveal converted coordinates | `bundle.get_matchstamp_at(coord, tl_id, conversion_maps=True)` |